In [19]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

#project modules
from src.data_loader import *
from src.features import *
from src.labeling import *
from src.volatility import *

In [9]:
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

In [10]:
df = add_returns(df)

df = add_close_to_close_volatility(df, window=20)

df = add_target_direction(df)
df = add_lagged_returns(df, lags=(1,5,10))
df = add_moving_average_features(df, windows=(5,10,20))
df = add_volatility_features(df, vol_col="vol_cc", lags=(1,))

df = df.dropna().reset_index(drop=True)
df.tail()

,Date,Open,High,Low,Close,Volume,return,vol_cc,target,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc_lag_1
4495,2026-02-13,25571.150391,25630.349609,25444.300781,25471.099609,453500,-0.013023,0.138191,1,-0.005650,0.001985,-0.003865,0.986987,0.991236,0.998418,0.130152
4496,2026-02-16,25423.599609,25697.000000,25372.699219,25682.750000,275800,0.008309,0.141515,1,-0.013023,0.006757,-0.009172,0.996614,0.997166,1.006737,0.138191
4497,2026-02-17,25637.949219,25764.400391,25570.300781,25725.400391,344100,0.001661,0.140728,1,0.008309,0.002623,0.025476,0.999897,0.998830,1.008133,0.141515
4498,2026-02-18,25752.650391,25828.050781,25645.150391,25819.349609,310200,0.003652,0.130728,0,0.001661,0.000721,0.001883,1.004599,1.002309,1.010652,0.140728
4499,2026-02-19,25873.349609,25885.300781,25388.750000,25454.349609,0,-0.014137,0.141139,0,0.003652,-0.005650,-0.005168,0.993124,0.988863,0.995786,0.130728


In [11]:
feature_cols = [
    "ret_lag_1",
    "ret_lag_5",
    "ret_lag_10",
    "ma_ratio_5",
    "ma_ratio_10",
    "ma_ratio_20",
    "vol_cc",
    "vol_cc_lag_1"
]

X = df[feature_cols]
y = df["target"]

In [12]:
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# **Logistic Regression**

In [14]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logit.fit(X_train_scaled, y_train)

pred_logit = logit.predict(X_test_scaled)

acc_logit = accuracy_score(y_test, pred_logit)
acc_logit

0.5444444444444444

# **Decision Tree**

In [15]:
tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)

tree.fit(X_train_scaled, y_train)

pred_tree = tree.predict(X_test_scaled)

acc_tree = accuracy_score(y_test, pred_tree)
acc_tree

0.5088888888888888

# **Random Forest**

In [16]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

pred_rf = rf.predict(X_test_scaled)

acc_rf = accuracy_score(y_test, pred_rf)
acc_rf

0.5633333333333334

# **Gradient Boosting**

In [17]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

gb.fit(X_train_scaled, y_train)

pred_gb = gb.predict(X_test_scaled)

acc_gb = accuracy_score(y_test, pred_gb)
acc_gb

0.5622222222222222

In [18]:
results = pd.DataFrame({
    "Model": ["Logistic", "DecisionTree", "RandomForest", "GradientBoost"],
    "Accuracy": [acc_logit, acc_tree, acc_rf, acc_gb]
})

results

,Model,Accuracy
0,Logistic,0.544444
1,DecisionTree,0.508889
2,RandomForest,0.563333
3,GradientBoost,0.562222
